# GCS kitchen navigation and pickup

In [1]:
import math
import threading
import time
from functools import partial

import plotly.graph_objects as go
import rclpy

from coraplex.datastructures.dataclasses import Context
from coraplex.datastructures.enums import Arms, ApproachDirection, VerticalAlignment
from coraplex.datastructures.grasp import GraspDescription
from coraplex.execution_environment import simulated_robot
from coraplex.plans.factories import code, sequential
from coraplex.robot_plans.actions.core.navigation import NavigateAction
from coraplex.robot_plans.actions.core.pick_up import PickUpAction
from coraplex.robot_plans.actions.core.robot_body import MoveTorsoAction, ParkArmsAction
from coraplex.testing import setup_world
from coraplex.view_manager import ViewManager
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from semantic_digital_twin.datastructures.definitions import TorsoState
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix, Point3, Pose
from semantic_digital_twin.world_description.geometry import BoundingBox
from semantic_digital_twin.world_description.graph_of_convex_sets import GraphOfConvexSets
from semantic_digital_twin.world_description.shape_collection import BoundingBoxCollection

In [2]:
world = setup_world()
robot = PR2.from_world(world)
robot.root.parent_connection.origin = HomogeneousTransformationMatrix.from_xyz_rpy(
    x=3.5,
    y=2.0,
    reference_frame=world.root,
)

milk = world.get_body_by_name("milk.stl")
cereal = world.get_body_by_name("breakfast_cereal.stl")

robot_position = robot.root.global_pose.to_position()
milk_position = milk.global_pose.to_position()
cereal_position = cereal.global_pose.to_position()

{
    "robot": tuple(
        round(float(value), 2)
        for value in (robot_position.x, robot_position.y, robot_position.z)
    ),
    "milk": tuple(
        round(float(value), 2)
        for value in (milk_position.x, milk_position.y, milk_position.z)
    ),
    "cereal": tuple(
        round(float(value), 2)
        for value in (cereal_position.x, cereal_position.y, cereal_position.z)
    ),
}

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


{'robot': (3.5, 2.0, 0.0),
 'milk': (2.37, 2.0, 1.05),
 'cereal': (2.37, 1.8, 1.05)}

In [3]:
if not rclpy.ok():
    rclpy.init()

ros_node = rclpy.create_node("coraplex_gcs_kitchen")
ros_thread = threading.Thread(target=rclpy.spin, args=(ros_node,), daemon=True)
ros_thread.start()

In [4]:
tf_publisher = TFPublisher(_world=world, node=ros_node)
visualization_publisher = VizMarkerPublisher(_world=world, node=ros_node)
print(f"RViz fixed frame: {world.root.name}")
print(f"MarkerArray topic: {visualization_publisher.topic_name}")

RViz fixed frame: apartment/apartment_root
MarkerArray topic: /semworld/viz_marker


In [5]:
context = Context(world=world, robot=robot, ros_node=ros_node)
context.evaluate_conditions = True

In [6]:
search_space = BoundingBoxCollection(
    [
        BoundingBox(
            min_x=0.7,
            max_x=4.2,
            min_y=0.2,
            max_y=4.8,
            min_z=0.15,
            max_z=0.25,
            origin=HomogeneousTransformationMatrix(reference_frame=world.root),
        )
    ],
    reference_frame=world.root,
)

graph_of_convex_sets = GraphOfConvexSets.free_space_from_world(
    world=world,
    search_space=search_space,
    bloat_obstacles=0.22,
)
graph_of_convex_sets.create_as_region(
    name=PrefixedName("kitchen_navigation_free_space")
)

start = Point3(3.5, 2.0, 0.2, reference_frame=world.root)
pickup_position = Point3(1.65, 2.0, 0.2, reference_frame=world.root)
raw_path = graph_of_convex_sets.path_from_to(start, pickup_position)
assert raw_path is not None and len(raw_path) > 2

minimum_waypoint_distance = 0.08
path = [raw_path[0]]
for candidate in raw_path[1:-1]:
    previous = path[-1]
    candidate_distance = math.hypot(
        float(candidate.x - previous.x),
        float(candidate.y - previous.y),
    )
    if candidate_distance >= minimum_waypoint_distance:
        path.append(candidate)
path.append(raw_path[-1])

{
    "graph_nodes": len(graph_of_convex_sets.graph.nodes()),
    "graph_edges": len(graph_of_convex_sets.graph.edges()),
    "raw_waypoints": len(raw_path),
    "filtered_path": [
        (round(float(point.x), 3), round(float(point.y), 3)) for point in path
    ],
}

INFO:graph_of_convex_sets.py::543 free_space_from_semantic_annotation Free space calculated in 22.245805 ms
INFO:graph_of_convex_sets.py::561 free_space_from_semantic_annotation Connectivity calculated in 20.575306 ms


{'graph_nodes': 72,
 'graph_edges': 72,
 'raw_waypoints': 23,
 'filtered_path': [(3.5, 2.0),
  (3.139, 0.636),
  (3.039, 0.635),
  (2.097, 0.635),
  (1.65, 2.0)]}

In [7]:
figure = go.Figure(graph_of_convex_sets.plot_free_space())
figure.add_trace(
    go.Scatter3d(
        x=[point.x for point in path],
        y=[point.y for point in path],
        z=[0.2 for point in path],
        mode="lines+markers",
        name="GCS path",
        line={"color": "black", "width": 8},
    )
)
figure.update_layout(height=600, scene_aspectmode="data")
figure

In [8]:
navigation_step_distance = 0.08
seconds_per_navigation_step = 0.08
navigation_poses = []

for segment_start, segment_end in zip(path, path[1:]):
    start_x = float(segment_start.x)
    start_y = float(segment_start.y)
    distance_x = float(segment_end.x) - start_x
    distance_y = float(segment_end.y) - start_y
    segment_distance = math.hypot(distance_x, distance_y)
    segment_step_count = max(
        1,
        math.ceil(segment_distance / navigation_step_distance),
    )
    segment_yaw = math.atan2(distance_y, distance_x)

    for segment_step_index in range(1, segment_step_count + 1):
        progress = segment_step_index / segment_step_count
        navigation_poses.append(
            Pose.from_xyz_rpy(
                x=start_x + distance_x * progress,
                y=start_y + distance_y * progress,
                z=0.0,
                yaw=segment_yaw,
                reference_frame=world.root,
            )
        )

pickup_yaw = math.atan2(
    float(milk.global_pose.y - pickup_position.y),
    float(milk.global_pose.x - pickup_position.x),
)
navigation_poses[-1] = Pose.from_xyz_rpy(
    x=pickup_position.x,
    y=pickup_position.y,
    z=0.0,
    yaw=pickup_yaw,
    reference_frame=world.root,
)

navigation_steps = []
for navigation_pose in navigation_poses:
    navigation_steps.extend(
        [
            NavigateAction(navigation_pose),
            code(partial(time.sleep, seconds_per_navigation_step)),
        ]
    )

len(navigation_poses)

50

In [9]:
grasp_description = GraspDescription(
    approach_direction=ApproachDirection.FRONT,
    vertical_alignment=VerticalAlignment.NoAlignment,
    end_effector=ViewManager.get_end_effector_view(Arms.LEFT, robot),
)

task = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        MoveTorsoAction(TorsoState.HIGH),
        *navigation_steps,
        PickUpAction(milk, Arms.LEFT, grasp_description),
    ],
    context=context,
)

with simulated_robot:
    task.perform()

In [10]:
left_end_effector = ViewManager.get_end_effector_view(Arms.LEFT, robot)
picked_up = world.get_connection(left_end_effector.tool_frame, milk) is not None
assert picked_up

{
    "graph_nodes": len(graph_of_convex_sets.graph.nodes()),
    "graph_edges": len(graph_of_convex_sets.graph.edges()),
    "filtered_waypoints": len(path),
    "navigation_updates": len(navigation_poses),
    "picked_up": picked_up,
    "cereal_remains_on_counter": cereal.parent_connection.parent == world.root,
}

{'graph_nodes': 72,
 'graph_edges': 72,
 'filtered_waypoints': 5,
 'navigation_updates': 50,
 'picked_up': True,
 'cereal_remains_on_counter': True}